# Player Intelligence System · CPE342

สมุดงานครบ 5 Task: ตรวจข้อมูล → ฝึกโมเดล → วิเคราะห์ข้อผิดพลาด → ตรวจไฟล์ส่ง

เริ่มจากอ่านผลที่มีอยู่ก่อน โดย `RUN_TRAINING=False` จะไม่ฝึกซ้ำ หากใช้บน Kaggle ครั้งแรกให้เปิด GPU และเปลี่ยนเป็น `True` เพื่อสร้างผลใหม่ ใช้ข้อมูลของการแข่งขันที่แนบไว้ใน Input และเปิด Internet เพื่อดาวน์โหลดน้ำหนัก ResNet18 ตามที่กติกา Task 4 อนุญาต หรือแนบน้ำหนักที่ดาวน์โหลดไว้แล้วและตั้ง TORCH_HOME

โค้ดทั้งหมดฝังอยู่ใน notebook นี้ จึงไม่ต้องแนบโฟลเดอร์ `pis` แยก เมื่อแก้ source ในเครื่องให้รัน `build_notebook.py` เพื่อสร้างฉบับที่ตรงกันอีกครั้ง

สิ่งที่ต้องรู้: Task 1 ขาด label 13,184 แถว ต้องแยกออก; Task 4 ใช้ชื่อเกมจาก CSV จริง; Task 5 ไม่มี label จึงไม่มีคะแนน F3 ในเครื่อง; การจับคู่ ANS กับแต่ละ Task ใช้ลำดับ test.csv ตามสมมติฐานที่ยังควรยืนยันกับ TA

อ้างอิง: [โจทย์ Kaggle](https://www.kaggle.com/competitions/player-intelligence-system-cpe-342-ml/overview/description) · [สไลด์ผู้สอน](https://www.canva.com/design/DAHVJwyZgy0/Ec3RMMLdy0FgfOLfIb3PhA/view)


## 1. ตั้งค่า

ในเครื่องให้เลือก Python จาก `solution/.venv` ถ้าขาดเครื่องมือให้ติดตั้ง `requirements.txt` ก่อน บน Kaggle มี PyTorch และ scikit-learn อยู่แล้ว หากไม่มี CatBoost โค้ดจะใช้ HistGradientBoosting แทน หรือรัน `%pip install catboost` ก่อนฝึก

`balanced` ใช้ CatBoost สูงสุด 1,000 รอบต่อ fold, ภาพ 5 epoch และ Isolation Forest 400 ต้นต่อมุมมอง `strong` เพิ่มงบฝึก แต่ไม่รับประกันว่าคะแนนดีขึ้น ต้องเปรียบเทียบบน split เดิม


In [ ]:
from pathlib import Path
import os, sys, json
import numpy as np
import pandas as pd

RUN_TRAINING = False
PROFILE = "balanced"
FOLDS = 3
SEED = 342
TASKS_TO_RUN = [1, 2, 3, 4, 5]
CONTAMINATION = 0.15  # สมมติฐานสัดส่วนผิดปกติ ยังไม่มี label ให้ปรับด้วย F3
DATA_ROOT = None  # กำหนด Path เองถ้าค้นเจอ dataset หลายชุด
IMAGE_ROOT = None  # ถ้าแนบภาพแยกจากตาราง ให้ชี้โฟลเดอร์ที่มี train.csv/test.csv ของภาพ

candidates = [Path.cwd(), Path.cwd()/"cpe342-kaggle-main", Path.cwd().parent/"cpe342-kaggle-main",
              Path.cwd()/"Player Intelligence System - CPE342 ML"/"cpe342-kaggle-main"]
if Path('/kaggle/input').exists():
    candidates += [p.parent for p in Path('/kaggle/input').rglob('sample_submission.csv')]
roots = sorted({p.resolve() for p in candidates if (p/'sample_submission.csv').exists()})
if DATA_ROOT is None:
    if len(roots) != 1:
        raise ValueError(f"ตั้ง DATA_ROOT ให้ชี้โฟลเดอร์ที่มี sample_submission.csv: พบ {roots}")
    DATA_ROOT = roots[0]
else:
    DATA_ROOT = Path(DATA_ROOT)
OUTPUT_DIR = (Path('/kaggle/working')/'outputs' if Path('/kaggle/working').exists()
              else DATA_ROOT.parent/'solution'/'outputs'/'main')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
cache_base = Path('/kaggle/working') if Path('/kaggle/working').exists() else OUTPUT_DIR.parent.parent
os.environ.setdefault('TORCH_HOME', str(cache_base/'.cache'/'torch'))
if IMAGE_ROOT is None:
    IMAGE_ROOT = DATA_ROOT/'04_image_classification'
    if not ((IMAGE_ROOT/'train.csv').exists() and
            ((IMAGE_ROOT/'train').is_dir() or (IMAGE_ROOT/'data'/'train').is_dir())) and Path('/kaggle/input').exists():
        image_roots = [p.parent for p in Path('/kaggle/input').rglob('train.csv')
                       if (p.parent/'test.csv').exists() and
                       ((p.parent/'train').is_dir() or (p.parent/'data'/'train').is_dir())]
        if len(image_roots)==1:
            IMAGE_ROOT=image_roots[0]
print('ข้อมูล:', DATA_ROOT, '\nผลทดลอง:', OUTPUT_DIR)
print('ภาพ:', IMAGE_ROOT)


In [ ]:
# Source snapshot: สร้าง package เดียวกับที่ใช้ทดสอบในเครื่อง
SOURCE_FILES = {'pis/__init__.py': '"""Reproducible CPE342 Player Intelligence experiments."""\n', 'pis/anomaly.py': '"""Task 5: unlabeled, transductive anomaly ranking. No invented F3 score."""\nfrom pathlib import Path\nimport re\nimport logging\nimport joblib\nimport numpy as np\nimport pandas as pd\nfrom sklearn.ensemble import IsolationForest\nfrom sklearn.impute import SimpleImputer\nfrom .common import write_json\n\nLOG = logging.getLogger(__name__)\n\n\ndef temporal_features(frame):\n    result = {}\n    families = sorted({re.sub(r\'_[1-4]$\', \'\', col) for col in frame if re.search(r\'_[1-4]$\', col)})\n    for family in families:\n        cols = [f\'{family}_{i}\' for i in range(1, 5)]\n        if not all(c in frame for c in cols):\n            continue\n        values = frame[cols]\n        history = values.iloc[:, :3]\n        mean = history.mean(axis=1)\n        delta = values.iloc[:, 3] - mean\n        result[f\'{family}_last_delta\'] = delta\n        result[f\'{family}_relative_delta\'] = delta / (mean.abs() + 1)\n        result[f\'{family}_range\'] = values.max(axis=1) - values.min(axis=1)\n        result[f\'{family}_first_last\'] = values.iloc[:, 3] - values.iloc[:, 0]\n    # Geographic discontinuity across successive observed periods (distance, not speed).\n    for i in range(1, 4):\n        cols = [f\'login_lat_{i}\', f\'login_lon_{i}\', f\'login_lat_{i+1}\', f\'login_lon_{i+1}\']\n        if all(c in frame for c in cols):\n            lat1, lon1, lat2, lon2 = [np.deg2rad(frame[c]) for c in cols]\n            a = np.sin((lat2-lat1)/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin((lon2-lon1)/2)**2\n            result[f\'location_distance_km_{i}_{i+1}\'] = 6371 * 2 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))\n    return pd.DataFrame(result, index=frame.index).replace([np.inf, -np.inf], np.nan)\n\n\ndef train_anomaly(data_root, output_dir, seed=342, profile=\'balanced\', contamination=0.15, threads=-1):\n    if not 0 < contamination < 0.5:\n        raise ValueError(\'contamination must be between 0 and 0.5\')\n    root, out = Path(data_root), Path(output_dir)\n    out.mkdir(parents=True, exist_ok=True)\n    frame = pd.read_csv(root / \'05_anomaly_detection\' / \'test.csv\')\n    if not frame[\'player_id\'].is_unique:\n        raise ValueError(\'Task 5 player_id must be unique\')\n    x = frame.drop(columns=\'player_id\').apply(pd.to_numeric, errors=\'raise\').replace([np.inf, -np.inf], np.nan)\n    if \'is_anomaly\' in x:\n        raise ValueError(\'Task 5 must remain unsupervised; found label column\')\n    trees = {\'quick\': 150, \'balanced\': 400, \'strong\': 800}[profile]\n    scores, artifacts = [], {}\n    for offset, (name, features) in enumerate(((\'raw\', x), (\'temporal\', temporal_features(x)))):\n        if features.shape[1] == 0:\n            continue\n        LOG.info(\'Task 5 fitting %s view: %s features\', name, features.shape[1])\n        imputer = SimpleImputer(strategy=\'median\', keep_empty_features=True)\n        values = imputer.fit_transform(features)\n        model = IsolationForest(n_estimators=trees, max_samples=min(4096, len(x)),\n                                contamination=\'auto\', random_state=seed + offset, n_jobs=threads)\n        model.fit(values)\n        raw = -model.score_samples(values)  # larger is more unusual\n        percentile = pd.Series(raw).rank(method=\'average\', pct=True).to_numpy()\n        scores.append(percentile)\n        artifacts[name] = {\'columns\': list(features.columns), \'imputer\': imputer, \'model\': model}\n    ensemble = np.mean(scores, axis=0)\n    # Stable ranking gives exactly ceil(n * fraction) flagged rows even when scores tie.\n    order = np.argsort(-ensemble, kind=\'stable\')\n    prediction = np.zeros(len(frame), dtype=int)\n    prediction[order[:int(np.ceil(contamination * len(frame)))]] = 1\n    pd.DataFrame({\'player_id\': frame.player_id, \'prediction\': prediction}).to_csv(out / \'task5_predictions.csv\', index=False)\n    scored = pd.DataFrame({\'player_id\': frame.player_id, \'anomaly_score\': ensemble, \'prediction\': prediction})\n    for name, view in zip(artifacts, scores):\n        scored[f\'{name}_rank\'] = view\n    scored.to_csv(out / \'task5_scores.csv\', index=False)\n    joblib.dump(artifacts, out / \'task5_models.joblib\')\n    metrics = {\'task\': 5, \'model\': \'IsolationForest raw+temporal percentile-rank ensemble\',\n               \'seed\': seed, \'profile\': profile, \'trees_per_view\': trees, \'n_test\': len(frame),\n               \'contamination_assumption\': contamination, \'flagged_rows\': int(prediction.sum()),\n               \'macro_f3\': None, \'metric_note\': \'No ground-truth labels; local F3 cannot be calculated.\',\n               \'fit_policy\': \'Task5-specific transductive unsupervised fit on provided unlabeled test.csv. General no-test-training rule needs TA clarification.\',\n               \'sensitivity_flagged_rows\': {str(q): int(np.ceil(q*len(frame))) for q in (.05,.1,.15,.2,.3)}}\n    write_json(out / \'task5_metrics.json\', metrics)\n    return metrics\n', 'pis/audit.py': '"""Inventory and data quality checks. Does not change the competition data."""\nfrom pathlib import Path\nimport hashlib\nimport pandas as pd\nfrom .common import TASKS, environment, write_json, image_folder\n\n\ndef audit_data(data_root, output_dir, image_root=None):\n    root, out = Path(data_root), Path(output_dir)\n    report = {\'environment\': environment(), \'tasks\': {}, \'warnings\': []}\n    template_path = root / \'sample_submission.csv\'\n    with template_path.open(encoding=\'utf-8-sig\') as handle:\n        header = handle.readline()\n    separator = \'|\' if \'|\' in header else \',\'\n    template = pd.read_csv(template_path, sep=separator)\n    report[\'submission\'] = {\'rows\': len(template), \'columns\': list(template.columns), \'separator\': separator,\n                            \'unique_ids\': bool(template[\'id\'].is_unique),\n                            \'sha256\': hashlib.sha256(template_path.read_bytes()).hexdigest()}\n    test_ids = {}\n    for number, (folder, target) in TASKS.items():\n        info = {}\n        task_root = root / folder\n        if number == 4 and image_root is not None:\n            task_root = image_folder(root, image_root)\n        for split in (\'train\', \'test\'):\n            path = task_root / f\'{split}.csv\'\n            if not path.exists():\n                continue\n            frame = pd.read_csv(path)\n            id_col = \'player_id\' if \'player_id\' in frame else \'id\' if \'id\' in frame else frame.columns[0]\n            item = {\'rows\': len(frame), \'columns\': list(frame.columns), \'id_column\': id_col,\n                    \'dtypes\': {col: str(dtype) for col, dtype in frame.dtypes.items()},\n                    \'unique_ids\': bool(frame[id_col].is_unique),\n                    \'missing_per_column\': {k: int(v) for k, v in frame.isna().sum().items() if v},\n                    \'sha256\': hashlib.sha256(path.read_bytes()).hexdigest()}\n            if target in frame:\n                y = frame[target]\n                item[\'missing_target_rows\'] = int(y.isna().sum())\n                if number == 3:\n                    item[\'target_summary\'] = {k: float(v) for k, v in y.describe().items()}\n                    item[\'target_zero_fraction\'] = float(y.eq(0).mean())\n                else:\n                    item[\'target_counts\'] = {str(k): int(v) for k, v in y.value_counts().items()}\n            if split == \'test\':\n                test_ids[number] = set(frame[id_col].astype(str))\n                item[\'matches_submission_row_count\'] = len(frame) == len(template)\n            info[split] = item\n        if number == 4:\n            files = [p for p in task_root.rglob(\'*\') if p.suffix.lower() in (\'.jpg\', \'.jpeg\', \'.png\')]\n            info[\'image_count\'] = len(files)\n            if files:\n                from PIL import Image\n                samples = []\n                for path in sorted(files)[:5]:\n                    with Image.open(path) as img:\n                        samples.append({\'file\': str(path.relative_to(task_root)),\n                                        \'width_px\': img.width, \'height_px\': img.height, \'mode\': img.mode})\n                info[\'image_sample\'] = samples\n            if not files or \'train\' not in info or \'test\' not in info:\n                report[\'warnings\'].append(\'Task 4 image dataset is incomplete; follow its README download link.\')\n        report[\'tasks\'][str(number)] = info\n    report[\'test_player_id_intersections\'] = {f\'{a}-{b}\': len(test_ids[a] & test_ids[b])\n                                             for a in test_ids for b in test_ids if a < b}\n    report[\'warnings\'].extend([\n        \'Submission ANS IDs have no supplied player-ID key. Assembly uses each task test.csv row order; confirm this convention with competition organizers.\',\n        \'Web text and local template disagree on separator and Task 4 label mapping. Template delimiter and actual training labels are authoritative locally; live evaluator is not verified.\',\n        \'Task 5 has only unlabeled test.csv. Its specific unsupervised task implies fitting that dataset, but this conflicts with the general no-test-training slide. Confirm the exception with the TA.\',\n    ])\n    write_json(out / \'data_audit.json\', report)\n    return report\n', 'pis/common.py': "from pathlib import Path\nimport json\nimport platform\nimport importlib.metadata\n\nTASKS = {\n    1: ('01_binary_classification', 'is_cheater'),\n    2: ('02_multiclass_classification', 'segment'),\n    3: ('03_regression', 'spending_30d'),\n    4: ('04_image_classification', 'class'),\n    5: ('05_anomaly_detection', None),\n}\nSOLUTION_ROOT = Path(__file__).resolve().parents[1]\nDEFAULT_DATA = SOLUTION_ROOT.parent / 'cpe342-kaggle-main'\nDEFAULT_OUTPUT = SOLUTION_ROOT / 'outputs' / 'main'\n\n\ndef image_folder(data_root, image_root=None):\n    base = Path(image_root) if image_root is not None else Path(data_root) / TASKS[4][0]\n    if (base/'train.csv').is_file() and (base/'test.csv').is_file():\n        return base\n    candidates = [p.parent for p in base.rglob('train.csv')\n                  if (p.parent/'test.csv').exists() and\n                  ((p.parent/'train').is_dir() or (p.parent/'data'/'train').is_dir())] if base.exists() else []\n    if len(candidates) == 1:\n        return candidates[0]\n    raise FileNotFoundError(f'Cannot locate unique Task 4 metadata/images under {base}; set --image-root explicitly.')\n\n\ndef write_json(path, obj):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(json.dumps(obj, indent=2, ensure_ascii=False, allow_nan=False), encoding='utf-8')\n\n\ndef environment():\n    packages = {}\n    for name in ('numpy', 'pandas', 'scikit-learn', 'catboost', 'torch', 'torchvision', 'Pillow'):\n        try:\n            packages[name] = importlib.metadata.version(name)\n        except importlib.metadata.PackageNotFoundError:\n            packages[name] = None\n    return {'python': platform.python_version(), 'platform': platform.platform(), 'packages': packages}\n", 'pis/image.py': '"""Train and evaluate the game screenshot classifier without using test labels."""\nfrom pathlib import Path\nimport hashlib\nimport json\nimport random\n\n\nclass _Images:\n    def __init__(self, paths, y, transform):\n        self.paths, self.y, self.transform = paths, y, transform\n\n    def __len__(self):\n        return len(self.paths)\n\n    def __getitem__(self, idx):\n        from PIL import Image\n        with Image.open(self.paths[idx]) as img:\n            x = self.transform(img.convert(\'RGB\'))\n        return x, (-1 if self.y is None else int(self.y[idx]))\n\n\ndef train_image(data_root: Path, output_dir: Path, seed: int = 342,\n                profile: str = \'balanced\', batch_size: int = 128,\n                epochs: int | None = None, workers: int = 0,\n                pretrained: bool = True) -> dict:\n    """Train ResNet18 on a duplicate-grouped split and predict the test set.\n\n    CSV labels are preserved in the output; internal class indices are never\n    interpreted as game names. No test image is used for training or selection.\n    """\n    import numpy as np\n    import pandas as pd\n    from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix\n    from sklearn.model_selection import StratifiedGroupKFold\n    import torch\n    from torch import nn\n    from torch.utils.data import DataLoader\n    from torchvision import transforms, models\n\n    data_root, output_dir = Path(data_root), Path(output_dir)\n    if profile not in {\'quick\', \'balanced\', \'strong\'}:\n        raise ValueError(\'profile must be quick, balanced, or strong\')\n    epochs = epochs if epochs is not None else {\'quick\': 2, \'balanced\': 5, \'strong\': 10}[profile]\n    if epochs < 1 or batch_size < 1 or workers < 0:\n        raise ValueError(\'epochs and batch_size must be positive and workers nonnegative\')\n    if not (data_root / \'train.csv\').is_file():\n        candidates = list(data_root.rglob(\'train.csv\')) if data_root.exists() else []\n        candidates = [p for p in candidates if (p.parent / \'test.csv\').is_file()\n                      and ((p.parent / \'data\' / \'train\').is_dir() or (p.parent / \'train\').is_dir())]\n        if len(candidates) != 1:\n            raise FileNotFoundError(f\'Expected a unique image dataset with train.csv, test.csv and data/train under {data_root}\')\n        data_root = candidates[0].parent\n    for name in (\'train.csv\', \'test.csv\'):\n        if not (data_root / name).is_file():\n            raise FileNotFoundError(f\'Missing image metadata: {data_root / name}\')\n    train = pd.read_csv(data_root / \'train.csv\', dtype={\'id\': str, \'player_id\': str})\n    test = pd.read_csv(data_root / \'test.csv\', dtype={\'id\': str, \'player_id\': str})\n    def choose(columns, options, what):\n        found = [c for c in options if c in columns]\n        if not found:\n            raise ValueError(f\'Missing {what}; expected one of {options}, got {list(columns)}\')\n        return found[0]\n    id_col = choose(train.columns, (\'id\', \'player_id\', \'image_id\', \'filename\', \'file\', \'image\'), \'image identifier\')\n    if id_col not in test.columns:\n        raise ValueError(f\'Test metadata lacks identifier column {id_col}\')\n    if train[id_col].duplicated().any() or test[id_col].duplicated().any():\n        raise ValueError(\'Training and test image identifiers must be unique within each file\')\n    label_col = choose(train.columns, (\'class\', \'label\', \'target\', \'prediction\'), \'label\')\n    if train[id_col].isna().any() or test[id_col].isna().any() or train[label_col].isna().any():\n        raise ValueError(\'Image identifiers and training labels must not be missing\')\n    classes = sorted(train[label_col].unique().tolist(), key=lambda x: (str(type(x)), str(x)))\n    if len(classes) != 5:\n        raise ValueError(f\'Expected exactly 5 training classes, found {len(classes)}\')\n    class_to_index = {v: i for i, v in enumerate(classes)}\n    labels = train[label_col].map(class_to_index).to_numpy(dtype=int)\n\n    def image_paths(frame, split):\n        folder = data_root / \'data\' / split\n        if not folder.is_dir():\n            folder = data_root / split\n        if not folder.is_dir():\n            raise FileNotFoundError(f\'Missing image folder: {folder}\')\n        paths = []\n        for value in frame[id_col]:\n            name = str(value)\n            path = folder / name\n            if not path.suffix:\n                path = folder / (name + \'.jpg\')\n            if not path.is_file():\n                raise FileNotFoundError(f\'Missing {split} image for {value}: {path}\')\n            paths.append(path)\n        return paths\n    train_paths, test_paths = image_paths(train, \'train\'), image_paths(test, \'test\')\n    # Duplicate bytes must remain on the same side of the validation split.\n    hashes = []\n    for path in train_paths:\n        with path.open(\'rb\') as handle:\n            hashes.append(hashlib.sha256(handle.read()).hexdigest())\n    splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)\n    fit_idx, val_idx = next(splitter.split(np.zeros(len(labels)), labels, groups=hashes))\n    if len(set(labels[fit_idx])) != 5 or len(set(labels[val_idx])) != 5:\n        raise ValueError(\'Cannot construct a grouped holdout containing all five classes\')\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n    device = torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\')\n    torch.backends.cudnn.deterministic = True\n    torch.backends.cudnn.benchmark = False\n    normalize = transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))\n    train_transform = transforms.Compose([transforms.Resize((128, 256)),\n        transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),\n        transforms.ToTensor(), normalize])\n    eval_transform = transforms.Compose([transforms.Resize((128, 256)), transforms.ToTensor(), normalize])\n    def loader(paths, y, transform, shuffle=False):\n        return DataLoader(_Images(paths, y, transform), batch_size=batch_size, shuffle=shuffle,\n                          num_workers=workers, pin_memory=device.type == \'cuda\',\n                          generator=torch.Generator().manual_seed(seed))\n    fit_loader = loader([train_paths[i] for i in fit_idx], labels[fit_idx], train_transform, True)\n    val_loader = loader([train_paths[i] for i in val_idx], labels[val_idx], eval_transform)\n    test_loader = loader(test_paths, None, eval_transform)\n    weights = models.ResNet18_Weights.DEFAULT if pretrained else None\n    model = models.resnet18(weights=weights)\n    model.fc = nn.Linear(model.fc.in_features, 5)\n    model.to(device)\n    optimizer = torch.optim.AdamW([\n        {\'params\': [p for n, p in model.named_parameters() if not n.startswith(\'fc.\')], \'lr\': 3e-5},\n        {\'params\': model.fc.parameters(), \'lr\': 3e-4}], weight_decay=1e-4)\n    criterion = nn.CrossEntropyLoss()\n    scaler = torch.amp.GradScaler(\'cuda\', enabled=device.type == \'cuda\')\n    output_dir.mkdir(parents=True, exist_ok=True)\n    (output_dir / \'models\').mkdir(parents=True, exist_ok=True)\n    checkpoint = output_dir / \'models\' / \'task4_best.pt\'\n    best, history = -1.0, []\n    def infer(data_loader):\n        model.eval()\n        predictions = []\n        with torch.inference_mode():\n            for x, _ in data_loader:\n                x = x.to(device, non_blocking=True)\n                with torch.autocast(device_type=device.type, enabled=device.type == \'cuda\'):\n                    logits = model(x)\n                predictions.extend(logits.argmax(1).cpu().tolist())\n        return np.asarray(predictions, dtype=int)\n    for epoch in range(epochs):\n        # Keep frozen BatchNorm statistics unchanged during the head-only epoch.\n        for name, param in model.named_parameters():\n            param.requires_grad = epoch > 0 or name.startswith(\'fc.\')\n        model.train()\n        if epoch == 0:\n            for module in model.modules():\n                if isinstance(module, nn.BatchNorm2d):\n                    module.eval()\n        loss_sum = 0.0\n        for x, y in fit_loader:\n            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)\n            optimizer.zero_grad(set_to_none=True)\n            with torch.autocast(device_type=device.type, enabled=device.type == \'cuda\'):\n                loss = criterion(model(x), y)\n            scaler.scale(loss).backward()\n            scaler.step(optimizer)\n            scaler.update()\n            loss_sum += loss.item() * len(y)\n        predicted = infer(val_loader)\n        score = f1_score(labels[val_idx], predicted, average=\'macro\', zero_division=0)\n        history.append({\'epoch\': epoch + 1, \'train_loss\': loss_sum / len(fit_idx), \'validation_macro_f1\': float(score)})\n        print(f\'Task 4 epoch {epoch + 1}/{epochs}: validation macro F1={score:.4f}\', flush=True)\n        if score > best:\n            best = score\n            torch.save({\'state_dict\': model.state_dict(), \'classes\': classes, \'seed\': seed,\n                        \'pretrained\': pretrained, \'epoch\': epoch + 1}, checkpoint)\n    saved = torch.load(checkpoint, map_location=device, weights_only=False)\n    model.load_state_dict(saved[\'state_dict\'])\n    val_pred, test_pred = infer(val_loader), infer(test_loader)\n    def original(indices):\n        return [classes[int(i)] for i in indices]\n    pd.DataFrame({\'player_id\': train.iloc[val_idx][id_col].to_numpy(),\n                  \'actual\': train.iloc[val_idx][label_col].to_numpy(),\n                  \'prediction\': original(val_pred)}).to_csv(output_dir / \'task4_validation.csv\', index=False)\n    pd.DataFrame({\'player_id\': test[id_col].to_numpy(), \'prediction\': original(test_pred)}).to_csv(\n        output_dir / \'task4_predictions.csv\', index=False)\n    metrics = {\'task\': 4, \'seed\': seed, \'profile\': profile, \'device\': str(device),\n               \'pretrained\': pretrained, \'train_size\': int(len(fit_idx)), \'validation_size\': int(len(val_idx)),\n               \'split\': \'stratified 5-fold grouped by SHA256 image bytes; first fold held out\',\n               \'classes\': [v.item() if hasattr(v, \'item\') else v for v in classes],\n               \'best_epoch\': int(saved[\'epoch\']), \'validation_macro_f1\': float(best),\n               \'validation_accuracy\': float(accuracy_score(labels[val_idx], val_pred)),\n               \'classification_report\': classification_report(labels[val_idx], val_pred, labels=list(range(5)),\n                                                                target_names=[str(v) for v in classes],\n                                                                output_dict=True, zero_division=0),\n               \'confusion_matrix\': confusion_matrix(labels[val_idx], val_pred, labels=list(range(5))).tolist(),\n               \'history\': history}\n    with (output_dir / \'task4_metrics.json\').open(\'w\', encoding=\'utf-8\') as handle:\n        json.dump(metrics, handle, ensure_ascii=False, indent=2)\n    return metrics\n', 'pis/submission.py': '"""Submission assembly: align within each task, never join different player pools."""\nfrom pathlib import Path\nimport hashlib\nimport numpy as np\nimport pandas as pd\nfrom .common import TASKS, write_json, image_folder\n\n\ndef read_template(data_root):\n    path = Path(data_root) / \'sample_submission.csv\'\n    with path.open(encoding=\'utf-8-sig\') as handle:\n        header = handle.readline()\n    sep = \'|\' if \'|\' in header else \',\'\n    sample = pd.read_csv(path, sep=sep)\n    expected = [\'id\'] + [f\'task{i}\' for i in range(1, 6)]\n    if list(sample.columns) != expected or not sample.id.is_unique or sample.id.isna().any():\n        raise ValueError(\'Invalid template schema or IDs\')\n    return sample, sep\n\n\ndef _ids(frame):\n    for col in (\'player_id\', \'id\', \'image_id\', \'filename\'):\n        if col in frame:\n            return frame[col].astype(str)\n    raise ValueError(\'Cannot identify task ID column\')\n\n\ndef _allowed(data_root, task, image_root=None):\n    if task in (1, 5):\n        return {0, 1}\n    if task == 2:\n        return {0, 1, 2, 3}\n    folder, target = TASKS[task]\n    train = pd.read_csv(image_folder(data_root, image_root) / \'train.csv\')\n    col = next((c for c in (\'class\', \'game_title\', \'label\', \'target\') if c in train), None)\n    if col is None:\n        raise ValueError(\'Cannot determine Task 4 label column\')\n    return set(train[col].dropna().unique())\n\n\ndef validate_frame(frame, data_root, image_root=None):\n    template, _ = read_template(data_root)\n    if list(frame.columns) != list(template.columns) or len(frame) != len(template):\n        raise ValueError(\'Submission column order or row count differs from template\')\n    if frame.isna().any().any():\n        raise ValueError(\'Submission contains missing predictions\')\n    if frame.id.astype(str).tolist() != template.id.astype(str).tolist():\n        raise ValueError(\'Submission IDs or their order differ from template\')\n    for task in (1, 2, 4, 5):\n        allowed = _allowed(data_root, task, image_root)\n        if not set(frame[f\'task{task}\'].unique()) <= allowed:\n            raise ValueError(f\'Task {task} has invalid labels; expected {allowed}\')\n    spending = pd.to_numeric(frame.task3, errors=\'raise\').to_numpy()\n    if not np.isfinite(spending).all() or (spending < 0).any():\n        raise ValueError(\'Task 3 predictions must be finite, nonnegative THB\')\n    return {\'rows\': len(frame), \'columns\': list(frame.columns), \'valid\': True}\n\n\ndef assemble(data_root, output_dir, image_root=None):\n    root, out = Path(data_root), Path(output_dir)\n    sample, sep = read_template(root)\n    result = sample[[\'id\']].copy()\n    sources = {}\n    # Fail before writing a submit-ready file if any task is absent.\n    for task, (folder, _) in TASKS.items():\n        path = out / f\'task{task}_predictions.csv\'\n        if not path.exists():\n            raise FileNotFoundError(f\'Missing {path.name}; train Task {task} first. No final submission was written.\')\n        prediction = pd.read_csv(path, dtype={\'player_id\': str})\n        task_root = image_folder(root, image_root) if task == 4 else root / folder\n        test = pd.read_csv(task_root / \'test.csv\')\n        ids = _ids(test)\n        if len(ids) != len(sample) or not ids.is_unique:\n            raise ValueError(f\'Task {task}: wrong test count or duplicate IDs\')\n        if not {\'player_id\', \'prediction\'} <= set(prediction.columns) or not prediction.player_id.is_unique:\n            raise ValueError(f\'Task {task}: invalid prediction ID schema\')\n        if set(ids) != set(prediction.player_id) or len(prediction) != len(ids):\n            raise ValueError(f\'Task {task}: prediction IDs do not match this task test.csv\')\n        values = prediction.set_index(\'player_id\').loc[ids, \'prediction\'].to_numpy()\n        if task != 3 and pd.api.types.is_numeric_dtype(values):\n            if not np.isfinite(values).all() or not np.equal(values, np.floor(values)).all():\n                raise ValueError(f\'Task {task}: fractional or nonfinite class label\')\n            values = values.astype(int)\n        result[f\'task{task}\'] = values\n        sources[f\'task{task}\'] = {\'prediction_sha256\': hashlib.sha256(path.read_bytes()).hexdigest(),\n                                  \'test_sha256\': hashlib.sha256((task_root/\'test.csv\').read_bytes()).hexdigest()}\n    check = validate_frame(result, root, image_root)\n    path = out / \'submission.csv\'\n    result.to_csv(path, index=False, sep=sep)\n    validate_frame(pd.read_csv(path, sep=sep), root, image_root)\n    manifest = {**check, \'separator\': sep, \'file\': str(path.resolve()), \'sources\': sources,\n                \'sha256\': hashlib.sha256(path.read_bytes()).hexdigest(),\n                \'alignment\': \'ANS row i uses row i of each task test.csv. Inferred convention; no explicit ANS/player mapping supplied.\',\n                \'scope\': \'Local format checks passed; not uploaded to Kaggle and evaluator acceptance unverified.\'}\n    write_json(out / \'submission_manifest.json\', manifest)\n    return manifest\n\n\ndef validate_file(path, data_root, image_root=None):\n    _, sep = read_template(data_root)\n    return validate_frame(pd.read_csv(path, sep=sep), data_root, image_root)\n', 'pis/tabular.py': '"""Leakage-aware cross-validation for the three tabular competition tasks."""\nfrom pathlib import Path\nimport json\nimport logging\nimport os\n\nimport joblib\nimport numpy as np\nimport pandas as pd\nfrom sklearn.compose import ColumnTransformer\nfrom sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor\nfrom sklearn.impute import SimpleImputer\nfrom sklearn.metrics import (classification_report, confusion_matrix, f1_score,\n                             fbeta_score, mean_absolute_error)\nfrom sklearn.model_selection import KFold, StratifiedKFold\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.preprocessing import OrdinalEncoder\nfrom threadpoolctl import threadpool_limits\n\nLOG = logging.getLogger(__name__)\nTASKS = {1: (\'01_binary_classification\', \'is_cheater\'),\n         2: (\'02_multiclass_classification\', \'segment\'),\n         3: (\'03_regression\', \'spending_30d\')}\nITERATIONS = {\'quick\': 250, \'balanced\': 1000, \'strong\': 2000}\n\n\ndef _score(y, prediction):\n    denominator = float(np.sum(y))\n    if denominator <= 0:\n        raise ValueError(\'Task 3 score requires a positive sum of targets\')\n    return 1.0 / (1.0 + float(np.abs(np.asarray(y) - prediction).sum()) / denominator)\n\n\ndef _threshold(y, probabilities):\n    # At each distinct probability include every observation with probability >= threshold.\n    order = np.argsort(-probabilities, kind=\'stable\')\n    p, target = probabilities[order], np.asarray(y)[order].astype(int)\n    ends = np.r_[np.flatnonzero(p[:-1] != p[1:]), len(p) - 1]\n    tp = np.cumsum(target)[ends].astype(float)\n    fp = (ends + 1) - tp\n    fn = target.sum() - tp\n    tn = len(target) - tp - fp - fn\n    pos = np.divide(5 * tp, 5 * tp + 4 * fn + fp, out=np.zeros_like(tp), where=(5 * tp + 4 * fn + fp) != 0)\n    neg = np.divide(5 * tn, 5 * tn + 4 * fp + fn, out=np.zeros_like(tn), where=(5 * tn + 4 * fp + fn) != 0)\n    scores = (pos + neg) / 2\n    none_score = fbeta_score(target, np.zeros_like(target), beta=2, average=\'macro\', zero_division=0)\n    i = int(np.argmax(scores))\n    return (float(np.nextafter(p[0], np.inf)), float(none_score)) if none_score > scores[i] else (float(p[ends[i]]), float(scores[i]))\n\n\ndef _prepare(train, test, target):\n    if \'player_id\' not in train or \'player_id\' not in test or target not in train:\n        raise ValueError(\'Missing player_id or training target\')\n    if target in test:\n        raise ValueError(\'Test data must not include the target\')\n    columns = [c for c in train.columns if c not in (\'player_id\', target)]\n    if set(columns) != set(test.columns) - {\'player_id\'}:\n        raise ValueError(\'Training and test feature columns differ\')\n    x, xt = train[columns].copy(), test[columns].copy()\n    categorical = [c for c in columns if not pd.api.types.is_numeric_dtype(x[c])]\n    for c in categorical:\n        for frame in (x, xt):\n            frame[c] = frame[c].astype(\'string\').fillna(\'__MISSING__\').astype(str)\n    for c in set(columns) - set(categorical):\n        for frame in (x, xt):\n            frame[c] = pd.to_numeric(frame[c], errors=\'raise\').replace([np.inf, -np.inf], np.nan)\n    return x, xt, categorical\n\n\ndef train_task(task: int, data_root: Path, output_dir: Path, folds: int = 3,\n               seed: int = 342, profile: str = \'balanced\', threads: int = -1,\n               engine: str = \'auto\') -> dict:\n    """Train fold models, write OOF/test predictions, models, and metrics.\n\n    Task 1\'s threshold is selected on the same OOF labels used to report its\n    tuned score, which makes that score optimistic. Fixed-threshold score is\n    reported separately. Test predictions average fold probabilities/values.\n    """\n    if task not in TASKS or profile not in ITERATIONS or engine not in (\'auto\', \'catboost\', \'sklearn\'):\n        raise ValueError(\'Invalid task, profile, or engine\')\n    if folds < 2 or threads == 0 or threads < -1:\n        raise ValueError(\'folds must be >= 2 and threads must be -1 or positive\')\n    try:\n        from catboost import CatBoostClassifier, CatBoostRegressor\n        has_catboost = True\n    except ImportError:\n        has_catboost = False\n    if engine == \'catboost\' and not has_catboost:\n        raise ImportError(\'CatBoost requested but not installed\')\n    chosen = \'catboost\' if has_catboost and engine != \'sklearn\' else \'sklearn\'\n    folder, target = TASKS[task]\n    data_root, output_dir = Path(data_root), Path(output_dir)\n    output_dir.mkdir(parents=True, exist_ok=True)\n    train = pd.read_csv(data_root / folder / \'train.csv\')\n    test = pd.read_csv(data_root / folder / \'test.csv\')\n    if target not in train:\n        raise ValueError(\'Missing training target\')\n    n_train_total = len(train)\n    unlabeled = train.loc[train[target].isna(), [\'player_id\']].copy()\n    unlabeled.to_csv(output_dir / f\'task{task}_unlabeled_ids.csv\', index=False)\n    train = train.loc[train[target].notna()].reset_index(drop=True)\n    if train.empty:\n        raise ValueError(\'No labeled training rows\')\n    x, xt, categorical = _prepare(train, test, target)\n    y = train[target].to_numpy()\n    if task == 3 and (np.any(y < 0) or np.sum(y) <= 0):\n        raise ValueError(\'Task 3 targets must be nonnegative with positive sum\')\n    classes = np.unique(y) if task != 3 else None\n    if task == 1 and not np.array_equal(classes, [0, 1]):\n        raise ValueError(\'Task 1 requires binary target values 0 and 1\')\n    if task != 3 and (len(classes) < 2 or min(np.unique(y, return_counts=True)[1]) < folds):\n        raise ValueError(\'Each class must have at least folds observations\')\n    splitter = (StratifiedKFold(n_splits=folds, shuffle=True, random_state=seed) if task != 3\n                else KFold(n_splits=folds, shuffle=True, random_state=seed))\n    n_outputs = len(classes) if task != 3 else 1\n    oof = np.zeros((len(train), n_outputs)); test_preds = np.zeros((len(test), n_outputs))\n    baseline = np.empty(len(train), dtype=y.dtype)\n    importances = []\n    fold_scores, best_iterations = [], []\n    iters = ITERATIONS[profile]\n    nthreads = os.cpu_count() or 1 if threads == -1 else threads\n    with threadpool_limits(limits=nthreads):\n        for fold, (tr, va) in enumerate(splitter.split(x, y if task != 3 else None), 1):\n            LOG.info(\'Task %s fold %s/%s (%s)\', task, fold, folds, chosen)\n            baseline[va] = (pd.Series(y[tr]).mode().iloc[0] if task != 3 else np.median(y[tr]))\n            if chosen == \'catboost\':\n                params = dict(iterations=iters, depth=6, learning_rate=0.05, random_seed=seed + fold,\n                              thread_count=threads, verbose=False, allow_writing_files=False,\n                              early_stopping_rounds=80)\n                if task == 3:\n                    model = CatBoostRegressor(loss_function=\'MAE\', **params)\n                else:\n                    model = CatBoostClassifier(loss_function=\'Logloss\' if task == 1 else \'MultiClass\', **params)\n                model.fit(x.iloc[tr], y[tr], cat_features=categorical, eval_set=(x.iloc[va], y[va]), use_best_model=True)\n                model.save_model(str(output_dir / f\'task{task}_fold{fold}.cbm\'))\n                importances.append(model.get_feature_importance())\n                best_iterations.append(int(model.get_best_iteration()) + 1)\n                val_x, test_x = x.iloc[va], xt\n            else:\n                numeric = [c for c in x.columns if c not in categorical]\n                preprocess = ColumnTransformer([\n                    (\'num\', SimpleImputer(strategy=\'median\', keep_empty_features=True), numeric),\n                    (\'cat\', OrdinalEncoder(handle_unknown=\'use_encoded_value\', unknown_value=-1), categorical),\n                ], remainder=\'drop\')\n                params = dict(max_iter=iters, learning_rate=0.05, max_leaf_nodes=31,\n                              early_stopping=True, n_iter_no_change=40, random_state=seed + fold)\n                estimator = (HistGradientBoostingRegressor(loss=\'absolute_error\', **params) if task == 3\n                             else HistGradientBoostingClassifier(**params))\n                model = Pipeline([(\'preprocess\', preprocess), (\'model\', estimator)])\n                model.fit(x.iloc[tr], y[tr])\n                best_iterations.append(int(model.named_steps[\'model\'].n_iter_))\n                joblib.dump(model, output_dir / f\'task{task}_fold{fold}.joblib\')\n                val_x, test_x = x.iloc[va], xt\n            if task == 3:\n                oof[va, 0] = np.maximum(0, model.predict(val_x))\n                test_preds[:, 0] += np.maximum(0, model.predict(test_x)) / folds\n                fold_scores.append(_score(y[va], oof[va, 0]))\n            else:\n                fold_classes = model.classes_ if chosen == \'catboost\' else model.named_steps[\'model\'].classes_\n                indices = [int(np.flatnonzero(fold_classes == c)[0]) for c in classes]\n                oof[va] = model.predict_proba(val_x)[:, indices]\n                test_preds += model.predict_proba(test_x)[:, indices] / folds\n                fold_pred = (oof[va, 1] >= 0.5).astype(int) if task == 1 else classes[oof[va].argmax(axis=1)]\n                fold_scores.append(float(fbeta_score(y[va], fold_pred, beta=2, average=\'macro\', zero_division=0) if task == 1 else f1_score(y[va], fold_pred, average=\'macro\', zero_division=0)))\n    metrics = {\'task\': task, \'engine\': chosen, \'profile\': profile, \'folds\': folds, \'seed\': seed,\n               \'threads\': threads, \'iterations_limit\': iters, \'n_train\': len(train), \'n_test\': len(test),\n               \'n_train_total\': n_train_total, \'n_labeled\': len(train), \'n_unlabeled_dropped\': len(unlabeled),\n               \'fold_scores\': fold_scores, \'fold_score_note\': \'Task 1 uses fixed threshold 0.5\',\n               \'best_iterations\': best_iterations}\n    oof_frame = pd.DataFrame({\'player_id\': train.player_id, \'target\': y})\n    if task == 1:\n        threshold, tuned = _threshold(y, oof[:, 1])\n        oof_prediction = (oof[:, 1] >= threshold).astype(int)\n        prediction = (test_preds[:, 1] >= threshold).astype(int)\n        metrics.update(metric=\'macro_f2\', oof_score=tuned, threshold=threshold,\n                       threshold_note=\'Threshold tuned and evaluated on the same OOF labels; tuned score is optimistic.\',\n                       fixed_0_5_oof_score=float(fbeta_score(y, oof[:, 1] >= 0.5, beta=2, average=\'macro\', zero_division=0)),\n                       baseline_score=float(fbeta_score(y, baseline, beta=2, average=\'macro\', zero_division=0)))\n        oof_frame[\'probability_1\'] = oof[:, 1]\n    elif task == 2:\n        oof_prediction, prediction = classes[oof.argmax(axis=1)], classes[test_preds.argmax(axis=1)]\n        metrics.update(metric=\'macro_f1\', oof_score=float(f1_score(y, oof_prediction, average=\'macro\', zero_division=0)),\n                       baseline_score=float(f1_score(y, baseline, average=\'macro\', zero_division=0)))\n        for i, label in enumerate(classes):\n            oof_frame[f\'probability_{label}\'] = oof[:, i]\n    else:\n        oof_prediction, prediction = oof[:, 0], test_preds[:, 0]\n        metrics.update(metric=\'1/(1+sum_absolute_error/sum_target)\', oof_score=_score(y, oof_prediction),\n                       mae=float(mean_absolute_error(y, oof_prediction)), baseline_score=_score(y, baseline),\n                       baseline_mae=float(mean_absolute_error(y, baseline)))\n    if task != 3:\n        metrics[\'classification_report\'] = classification_report(y, oof_prediction, output_dict=True, zero_division=0)\n        metrics[\'confusion_matrix\'] = confusion_matrix(y, oof_prediction, labels=classes).tolist()\n        metrics[\'confusion_matrix_labels\'] = [v.item() if isinstance(v, np.generic) else v for v in classes]\n        metrics[\'classification_report_note\'] = \'Task 1 uses OOF-tuned threshold and is optimistic.\' if task == 1 else \'OOF predictions\'\n        probability_frame = pd.DataFrame({\'player_id\': test.player_id})\n        for i, label in enumerate(classes):\n            probability_frame[f\'probability_{label}\'] = test_preds[:, i]\n        probability_frame.to_csv(output_dir / f\'task{task}_probabilities.csv\', index=False)\n    if importances:\n        metrics[\'feature_importance\'] = dict(sorted(zip(x.columns, np.mean(importances, axis=0).tolist()), key=lambda item: item[1], reverse=True))\n    oof_frame[\'prediction\'] = oof_prediction\n    oof_frame.to_csv(output_dir / f\'task{task}_oof.csv\', index=False)\n    pd.DataFrame({\'player_id\': test.player_id, \'prediction\': prediction}).to_csv(output_dir / f\'task{task}_predictions.csv\', index=False)\n    (output_dir / f\'task{task}_metrics.json\').write_text(json.dumps(metrics, indent=2, allow_nan=False), encoding=\'utf-8\')\n    return metrics\n'}
runtime = Path.cwd()/'notebook_runtime'
for name, content in SOURCE_FILES.items():
    path = runtime/name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding='utf-8')
sys.path.insert(0, str(runtime.resolve()))
from pis.audit import audit_data
from pis.tabular import train_task
from pis.anomaly import train_anomaly
from pis.image import train_image
from pis.submission import assemble, validate_file


## 2. สำรวจข้อมูลก่อนเลือกโมเดล

ดูจำนวนตัวอย่าง ค่าที่หายไป สัดส่วนคลาส และข้อมูลเป้าหมาย ห้ามใช้ `player_id` เป็นตัวทำนาย และห้ามนำผู้เล่นคนละชุดมาเชื่อมเพราะแต่ละ Task มีรหัสคนละลำดับ

Task 3 มีผู้ไม่ใช้จ่ายจำนวนมากและหางยาว ให้ดูทั้งสัดส่วนศูนย์และการกระจาย ไม่ใช้ค่าเฉลี่ยเพียงอย่างเดียวอธิบายผู้เล่นทุกคน


In [ ]:
audit = audit_data(DATA_ROOT, OUTPUT_DIR, IMAGE_ROOT)
display(pd.DataFrame([{'task':task, 'split':split, 'rows':item['rows'],
                       'missing_target':item.get('missing_target_rows',0),
                       'missing_cells':sum(item['missing_per_column'].values())}
                      for task,info in audit['tasks'].items()
                      for split,item in info.items() if split in ('train','test')]))
for note in audit['warnings']:
    print('ข้อสังเกต:', note)


In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1,3,figsize=(15,4))
for task, ax in zip((1,2),axes[:2]):
    target = 'is_cheater' if task==1 else 'segment'
    folder = '01_binary_classification' if task==1 else '02_multiclass_classification'
    y = pd.read_csv(DATA_ROOT/folder/'train.csv',usecols=[target])[target]
    y.value_counts().sort_index().plot.bar(ax=ax,color='#236c9c')
    ax.set(title=f'Task {task}: labeled training class counts',xlabel='Class',ylabel='Players (count)')
y3 = pd.read_csv(DATA_ROOT/'03_regression'/'train.csv',usecols=['spending_30d']).spending_30d
axes[2].hist(np.log1p(y3),bins=50,color='#236c9c')
axes[2].set(title='Task 3: training spending distribution',xlabel='log(1 + spending in THB)',ylabel='Players (count)')
fig.suptitle('Source: provided training CSVs; log transform is for visualization only')
plt.tight_layout()
plt.show()


## 3. ฝึกและตรวจสอบ Tasks 1–3

CatBoost รับทั้งค่าตัวเลขและหมวดหมู่ ช่วยลดขั้นตอนแปลงข้อมูล ใช้ cross-validation เพื่อให้ทุกแถวมีผลทำนายจากโมเดลที่ไม่ได้ฝึกบนแถวนั้น เรียกว่า OOF หากใช้โมเดลสำรอง การเติมค่าหายและแปลงหมวดหมู่จะเรียนรู้เฉพาะ fold ฝึก

Task 1 ใช้ macro F2: คำนวณ F2 แยกแต่ละคลาสแล้วเฉลี่ย ไม่เท่ากับ F2 เฉพาะคลาสโกง การเลือก threshold บน OOF แล้วรายงานคะแนนบน OOF เดิมมีความเอนเอียง คะแนน threshold 0.5 จึงแสดงไว้ด้วย

Task 2 ใช้ macro F1 เพื่อไม่ให้คลาสใหญ่กลบคลาสเล็ก

Task 3 ใช้คะแนน `1/(1 + sum(abs(y-yhat))/sum(y))` ยิ่งสูงยิ่งดี ถ้าทายศูนย์ทุกคนจะได้ 0.5 เมื่อยอดรวมจริงเป็นบวก ลด MAE บนหน่วยบาทจริงจึงตรงกับคะแนนนี้ การฝึก log-target แล้วแปลงกลับไม่รับประกันว่า MAE จะดีขึ้น


In [ ]:
if RUN_TRAINING:
    for task in (1,2,3):
        if task in TASKS_TO_RUN:
            result = train_task(task,DATA_ROOT,OUTPUT_DIR,folds=FOLDS,seed=SEED,
                                profile=PROFILE,threads=8)
            print('Task',task,'OOF:',result['oof_score'])
else:
    print('ใช้ผลเดิม; เปลี่ยน RUN_TRAINING=True เพื่อฝึกใหม่')


## 4. Task 4: ภาพเกม

ใช้ ResNet18 ที่ฝึกล่วงหน้าตามสิทธิ์ของ Task 4 แล้วปรับกับภาพการแข่งขัน แยกภาพที่มี byte เหมือนกันให้อยู่ฝั่งเดียวกัน เพื่อลดคะแนนตรวจสอบที่สูงจากภาพซ้ำ เริ่มฝึกชั้นทำนายก่อน แล้วจึงปรับทั้งโมเดล เก็บ checkpoint ที่ macro F1 สูงสุด

ผลนี้เป็นคะแนนจาก holdout เดียวที่ใช้เลือก epoch จึงอาจสูงกว่าผลใหม่ได้ ภาพใกล้เคียงกันแต่ byte ต่างกันยังอาจข้าม split ควรตรวจต่อหากคะแนนสูงผิดปกติ ใช้ป้ายกำกับชื่อเกมตาม CSV จริง ไม่มีการเดาลำดับคลาสจากหน้า Overview


In [ ]:
if RUN_TRAINING and 4 in TASKS_TO_RUN:
    result = train_image(IMAGE_ROOT,OUTPUT_DIR,seed=SEED,profile=PROFILE,batch_size=128,workers=0)
    print('Image holdout macro F1:',result['validation_macro_f1'])


## 5. Task 5: ความผิดปกติที่ไม่มีคำตอบกำกับ

ฝึก Isolation Forest สองมุมมอง ได้แก่ค่าพฤติกรรมดิบ และการเปลี่ยนแปลงข้าม 4 ช่วงเวลา รวมอันดับคะแนนเพื่อไม่ให้สเกลของโมเดลหนึ่งกลบอีกโมเดล การเลือก 15% ที่คะแนนสูงสุดเป็นสมมติฐานเริ่มต้น ไม่ใช่สัดส่วนความผิดปกติที่พิสูจน์แล้ว

งานนี้มีเฉพาะ test.csv ที่ไม่มี label จึงตีความตามโจทย์เฉพาะว่าใช้ข้อมูลนี้ฝึกแบบไม่มีผู้สอนได้ แต่สไลด์กติกาทั่วไปห้ามฝึกบน test ต้องยืนยันข้อยกเว้นกับ TA ก่อนส่งงานจริง คะแนน F3 จะทราบจากการประเมินที่มีคำตอบจริงเท่านั้น


In [ ]:
if RUN_TRAINING and 5 in TASKS_TO_RUN:
    result = train_anomaly(DATA_ROOT,OUTPUT_DIR,seed=SEED,profile=PROFILE,
                           contamination=CONTAMINATION,threads=8)
    print('Flagged:',result['flagged_rows'],'Local F3:',result['macro_f3'])


## 6. อ่านผลและตรวจข้อผิดพลาด

อย่าเทียบคะแนนคนละ Task ราวกับเป็น metric เดียวกัน และยังคำนวณคะแนนรวมที่เทียบ Kaggle ไม่ได้ เพราะไม่มีคำตอบ Task 5 และยังไม่ทราบน้ำหนักคะแนนที่แน่นอน ตรวจ confusion matrix ต่อคลาสและตัวอย่างที่ผิด แล้วเลือกการทดลองถัดไปจากหลักฐาน


In [ ]:
metrics = {}
for task in range(1,6):
    path = OUTPUT_DIR/f'task{task}_metrics.json'
    if path.exists():
        metrics[task] = json.loads(path.read_text(encoding='utf-8'))
display(pd.DataFrame([{'task':t, 'metric':m.get('metric','macro_f1' if t==4 else 'macro_f3'),
                        'score':m.get('fixed_0_5_oof_score',m.get('oof_score',m.get('validation_macro_f1'))),
                        'evaluation':'OOF fixed threshold' if t==1 else 'OOF' if t<4 else 'holdout' if t==4 else 'unlabeled'}
                       for t,m in metrics.items()]))
for task,m in metrics.items():
    if 'confusion_matrix' not in m:
        continue
    matrix=np.array(m['confusion_matrix'])
    fig,ax=plt.subplots(figsize=(5,4))
    im=ax.imshow(matrix,cmap='Blues')
    for (r,c),value in np.ndenumerate(matrix):
        ax.text(c,r,str(value),ha='center',va='center')
    caption = f"OOF, tuned threshold {m['threshold']:.3f}" if task==1 else 'OOF' if task==2 else 'image holdout'
    names = m.get('classes',list(range(len(matrix))))
    ax.set(title=f'Task {task}: confusion matrix ({caption})',xlabel='Predicted class',ylabel='True class')
    ax.set_xticks(range(len(names)),names,rotation=35,ha='right')
    ax.set_yticks(range(len(names)),names)
    fig.colorbar(im,ax=ax,label='Players/images (count)')
    plt.show()


### โมเดลใช้ข้อมูลใด และผิดพลาดกับใคร

ความสำคัญของคุณลักษณะช่วยเสนอสมมติฐานสำหรับทดลองต่อ แต่ไม่พิสูจน์สาเหตุทางพฤติกรรม ส่วน MAE แยกกลุ่มยอดใช้จ่ายช่วยเปิดเผยว่าคะแนนเฉลี่ยซ่อนความผิดพลาดของผู้เล่นยอดสูงหรือไม่


In [ ]:
for task,m in metrics.items():
    importance=m.get('feature_importance')
    if not importance:
        continue
    values=pd.Series(importance).sort_values().tail(10)
    fig,ax=plt.subplots(figsize=(8,4))
    values.plot.barh(ax=ax,color='#236c9c')
    ax.set(title=f'Task {task}: top 10 CatBoost feature importance',xlabel='Relative model importance (%)',ylabel='Input feature')
    fig.tight_layout()
    plt.show()
path=OUTPUT_DIR/'task3_oof.csv'
if path.exists():
    errors=pd.read_csv(path)
    errors['absolute_error_thb']=(errors.target-errors.prediction).abs()
    errors['actual_spending_band']=pd.cut(errors.target,[-1e-9,0,1000,10000,np.inf],
                                          labels=['zero','0-1,000 THB','1,000-10,000 THB','10,000+ THB'])
    display(errors.groupby('actual_spending_band',observed=True).agg(
        players=('target','size'),mae_thb=('absolute_error_thb','mean')))


## 7. สร้างไฟล์ส่งอย่างตรวจสอบได้

ต้องมีผลครบทั้ง 5 Task จึงสร้าง `submission.csv` โค้ดจะเช็กรหัสผู้เล่นภายในแต่ละ Task แล้วเรียงกลับตาม test.csv จากนั้นกำหนดให้แถวที่ i ตรงกับ ANS แถวที่ i โดยไม่ join ผู้เล่นข้าม Task ใช้ตัวคั่นตามไฟล์ตัวอย่างจริงและชื่อเกมจาก CSV

สมมติฐานลำดับ ANS, ชื่อเกมที่ตัวตรวจยอมรับ และข้อยกเว้น Task 5 ควรยืนยันกับผู้สอน การผ่านการตรวจในเครื่องยังไม่ใช่หลักฐานว่า Kaggle ยอมรับไฟล์หรือได้คะแนนเท่าใด Notebook นี้ไม่อัปโหลดคำตอบให้อัตโนมัติ


In [ ]:
missing = [t for t in range(1,6) if not (OUTPUT_DIR/f'task{t}_predictions.csv').exists()]
if missing:
    print('ยังไม่สร้างไฟล์ส่ง: ขาดผล Task',missing)
else:
    manifest=assemble(DATA_ROOT,OUTPUT_DIR,IMAGE_ROOT)
    print(validate_file(OUTPUT_DIR/'submission.csv',DATA_ROOT,IMAGE_ROOT))
    print('ไฟล์:',manifest['file'])
    display(pd.read_csv(OUTPUT_DIR/'submission.csv',sep=manifest['separator']).head())


## 8. บันทึกสิ่งที่เรียนรู้และแผนทดลอง

สำหรับแต่ละ Task จด 1) ปัญหาและ metric 2) วิธีแบ่งข้อมูล 3) การจัดการค่าหาย/คลาส 4) โมเดลและเหตุผล 5) ผลเทียบ baseline 6) ข้อผิดพลาดและผลทางธุรกิจ 7) สิ่งที่ทดลองแล้วดีหรือแย่ลง

การทดลองที่มีประโยชน์ถัดไป: ปรับน้ำหนักคลาส Task 1–2 บน split เดิม, เปรียบเทียบโมเดลสองขั้นใน Task 3 ด้วย MAE จริง, ตรวจภาพใกล้ซ้ำและปรับ augmentation ใน Task 4, สำรวจความไวต่อ contamination ของ Task 5 โดยไม่อ้างว่าคะแนนดีขึ้นหากไม่มี label

อ่านคำอธิบายและตัวอย่างคำนวณเพิ่มเติมใน `STUDY_GUIDE_TH.md` และเก็บผลทดลองทุกครั้งแยกโฟลเดอร์ ไม่ใช้ public leaderboard เป็นชุด validation หลัก
